# ភាគបន្ថែម D៖ ការបន្ថែមមុខងារទៅក្នុង Training Loop

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/appendix-D/01_main-chapter-code/appendix-D.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ការត្រៀមបរិស្ថាន
- ## ១. ការផ្ទុកទិន្នន័យ
- ## ២. LR warmup scheduler
- ## ៣. Cosine decay schedule
- ## ៤. Gradient clipping
- ## ៥. ការវាយតម្លៃអំឡុងពេល train
- ## ៦. ក្រាផ loss ចុងក្រោយ

In [ ]:
# ការតភ្ជាប់ឯកសារជំនួយ — ឱ្យដំណើរការភ្ជាប់ដូច notebook ដើមរបស់ upstream
# (ចង្អុល sys.path និងថតធ្វើការទៅកាន់ថតកូដក្នុងឃ្លាំងកសារខ្មែរនេះ)
import os, sys
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "កូដ")):
    _p = os.path.dirname(_p)
_UP = os.path.join(_p, "កូដ", "appendix-D/01_main-chapter-code")
sys.path.insert(0, _UP)
os.chdir(_UP)


## ០. ការត្រៀមបរិស្ថាន

ពិនិត្យកំណែបណ្ណាល័យ និងនាំចូលម៉ូឌុលចាំបាច់។

In [ ]:
from importlib.metadata import version
import torch

print("torch version:", version("torch"))


from previous_chapters import GPTModel
# បើឯកសារ `previous_chapters.py` មិនមានក្នុងមូលដ្ឋានទេ អ្នកអាច import វាពី package PyPI `llms-from-scratch`។
# សម្រាប់លម្អិត សូមមើល៖ https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# ឧទាហរណ៍៖
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,  # ទំហំ vocabulary
    "context_length": 256,  # ប្រវែង context ខ្លី (ដើម 1024)
    "emb_dim": 768,  # វិមាត្រ embedding
    "n_heads": 12,  # ចំនួន attention heads
    "n_layers": 12,  # ចំនួនស្រទាប់
    "drop_rate": 0.1,  # អត្រា dropout
    "qkv_bias": False  # bias សម្រាប់ query-key-value
}

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # ប្រើ PyTorch 2.9 ឬថ្មីជាងនេះ សម្រាប់លទ្ធផល mps ស្ថិរ
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # បិទ dropout ពេលធ្វើ inference

## ១. ការផ្ទុកទិន្នន័យ

បង្កើត train/val dataloaders ពីអត្ថបទឧទាហរណ់។

In [ ]:
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

# សៀវភៅដើមបានប្រើកូដខាងក្រោមនេះ
# ប៉ុន្តែ urllib ប្រើការកំណត់ protocol ចាស់ ដែលអាចបង្កបញ្ហាដល់អ្នកប្រើមួយចំនួនដែលប្រើ VPN។
# កំណែ `requests` ខាងលើមានភាពរឹងមាំជាង urllib ដែលជាទូទៅដំណើរការល្អ ប៉ុន្តែអ្នកអានខ្លះបានរាយការណ៍ពីបញ្ហា urllib ពេលប្រើ VPN (របស់ក្រុមហ៊ុន)។

"""
import os
import urllib.request

if not os.path.exists(file_path):
    with urllib.request.urlopen(url) as response:
        text_data = response.read().decode('utf-8')
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
"""

In [ ]:
from previous_chapters import create_dataloader_v1
# ជាជម្រើស៖
# from llms_from_scratch.ch02 import create_dataloader_v1


# សមាមាត់ train/validation
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    text_data[:split_idx],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    text_data[split_idx:],
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

## ២. LR warmup scheduler

សាងសង់ learning-rate warmup ដែលឡើងជាបន្តបន្ទាប់ដល់ peak។

In [ ]:
n_epochs = 15
initial_lr = 0.0001
peak_lr = 0.01

In [ ]:
total_steps = len(train_loader) * n_epochs
warmup_steps = int(0.2 * total_steps)  # warmup 20%
print(warmup_steps)

In [ ]:
lr_increment = (peak_lr - initial_lr) / warmup_steps

global_step = -1
track_lrs = []

optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.1)

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1
    
        if global_step < warmup_steps:
            lr = initial_lr + global_step * lr_increment
        else:
            lr = peak_lr
        
        # អនុវត្ត learning rate ដែលបានគណនាទៅលើ optimizer
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        track_lrs.append(optimizer.param_groups[0]["lr"])
    
        # គណនា loss ហើយធ្វើបច្ចុប្បន្ភាព weights
        # ...

## ៣. Cosine decay schedule

បន្ថែមការថយចុះបែប cosine បន្ទាប់ពី warmup។

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.ylabel("Learning rate")
plt.xlabel("Step")
total_training_steps = len(train_loader) * n_epochs
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout(); plt.savefig("1.pdf")
plt.show()

In [ ]:
import math

min_lr = 0.1 * initial_lr
track_lrs = []

lr_increment = (peak_lr - initial_lr) / warmup_steps
global_step = -1

for epoch in range(n_epochs):
    for input_batch, target_batch in train_loader:
        optimizer.zero_grad()
        global_step += 1
    
        # លៃតម្រូវ learning rate តាមដំណាក់កាលបច្ចុបន្ន (warmup ឬ cosine annealing)
        if global_step < warmup_steps:
            # warmup បែបលីនេអ៊ែរ
            lr = initial_lr + global_step * lr_increment  
        else:
            # cosine annealing បន្ទាប់ពី warmup
            progress = ((global_step - warmup_steps) / 
                        (total_training_steps - warmup_steps))
            lr = min_lr + (peak_lr - min_lr) * 0.5 * (
                1 + math.cos(math.pi * progress))
        
        # អនុវត្ត learning rate ដែលបានគណនាទៅលើ optimizer
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr
        track_lrs.append(optimizer.param_groups[0]["lr"])
    
        # គណនា loss ហើយធ្វើបច្ចុប្បន្ភាព weights

In [ ]:
plt.figure(figsize=(5, 3))
plt.ylabel("Learning rate")
plt.xlabel("Step")
plt.plot(range(total_training_steps), track_lrs)
plt.tight_layout(); plt.savefig("2.pdf")
plt.show()

## ៤. Gradient clipping

កាត់បន្ថយ gradient norm ដើម្បីស្ថិរភាពការបណ្តុះបណ្តាល។

In [ ]:
from previous_chapters import calc_loss_batch
# ជាជម្រើស៖
# from llms_from_scratch.ch05 import calc_loss_batch


torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

loss = calc_loss_batch(input_batch, target_batch, model, device)
loss.backward()

In [ ]:
def find_highest_gradient(model):
    max_grad = None
    for param in model.parameters():
        if param.grad is not None:
            grad_values = param.grad.data.flatten()
            max_grad_param = grad_values.max()
            if max_grad is None or max_grad_param > max_grad:
                max_grad = max_grad_param
    return max_grad

print(find_highest_gradient(model))

In [ ]:
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(find_highest_gradient(model))

## ៥. ការវាយតម្លៃអំឡុងពេល train

បង្កើតអត្ថបទឧទាហរណ៍ជាចង្វាក់ ដើម្បីមើលវឌ្ឍនភាពម៉ូដែល។

In [ ]:
from previous_chapters import evaluate_model, generate_and_print_sample
# ជាជម្រើស៖
# from llms_from_scratch.ch05 import evaluate_model, generate_and_print_samplee


ORIG_BOOK_VERSION = False


def train_model(model, train_loader, val_loader, optimizer, device,
                n_epochs, eval_freq, eval_iter, start_context, tokenizer,
                warmup_steps, initial_lr=3e-05, min_lr=1e-6):

    train_losses, val_losses, track_tokens_seen, track_lrs = [], [], [], []
    tokens_seen, global_step = 0, -1

    # ទាញយក learning rate អតិបរមាពី optimizer
    peak_lr = optimizer.param_groups[0]["lr"]

    # គណនាចំនួន iterations សរុបក្នុងដំណើរការបណ្តុះបណ្តាល
    total_training_steps = len(train_loader) * n_epochs

    # គណនាកំណើន learning rate អំឡុងដំណាក់កាល warmup
    lr_increment = (peak_lr - initial_lr) / warmup_steps

    for epoch in range(n_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            global_step += 1

            # លៃតម្រូវ learning rate តាមដំណាក់កាលបច្ចុបន្ន (warmup ឬ cosine annealing)
            if global_step < warmup_steps:
                # warmup បែបលីនេអ៊ែរ
                lr = initial_lr + global_step * lr_increment  
            else:
                # cosine annealing បន្ទាប់ពី warmup
                progress = ((global_step - warmup_steps) / 
                            (total_training_steps - warmup_steps))
                lr = min_lr + (peak_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * progress))

            # អនុវត្ត learning rate ដែលបានគណនាទៅលើ optimizer
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr
            track_lrs.append(lr)  # រក្សាទុក learning rate បច្ចុបន្ន

            # គណនា និង backpropagate loss
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()

            # អនុវត្ត gradient clipping បន្ទាប់ពី warmup ដើម្បីចៀសវាង gradients ផ្ទុះ
            if ORIG_BOOK_VERSION:
                if global_step > warmup_steps:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  
            else:
                if global_step >= warmup_steps:  # the book originally used global_step > warmup_steps, which led to a skipped clipping step after warmup
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
            optimizer.step()
            tokens_seen += input_batch.numel()

            # វាយតម្លៃម៉ូដែលជាចង្វាក់លើសំណុំ train និង validation
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader,
                    device, eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                # បង្ហាញ losses បច្ចុបន្ន
                print(f"Ep {epoch+1} (Iter {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}"
                )

        # បង្កើត និងបង្ហាញគំរូពីម៉ូដែល ដើម្បីតាមដានវឌ្ឍនភាព
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    return train_losses, val_losses, track_tokens_seen, track_lrs

In [ ]:
import tiktoken

# Note:
# ដោះ comment កូដខាងក្រោមដើម្បីគណនារយៈពេលដំណើរការ
# import time
# start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)

peak_lr = 0.001  # នេះត្រូវបានកំណត់ដំបូងជា 5e-4 ក្នុងសៀវភៅដោយច្រឡំ
optimizer = torch.optim.AdamW(model.parameters(), lr=peak_lr, weight_decay=0.1)  # សៀវភៅបានបំភ្លេចការផ្តល់តម្លៃ lr ដោយអចេតនា
tokenizer = tiktoken.get_encoding("gpt2")

n_epochs = 15
train_losses, val_losses, tokens_seen, lrs = train_model(
    model, train_loader, val_loader, optimizer, device, n_epochs=n_epochs,
    eval_freq=5, eval_iter=1, start_context="Every effort moves you",
    tokenizer=tokenizer, warmup_steps=warmup_steps, 
    initial_lr=1e-5, min_lr=1e-5
)

# Note:
# ដោះ comment កូដខាងក្រោមដើម្បីបង្ហាញរយៈពេលដំណើរការ
# end_time = time.time()
# execution_time_minutes = (end_time - start_time) / 60
# print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(range(len(lrs)), lrs)
plt.ylabel("Learning rate")
plt.xlabel("Steps")
plt.show()

## ៦. ក្រាផ loss ចុងក្រោយ

គូសក្រាផ loss curves ដែលបានកែលម្អ។

In [ ]:
from previous_chapters import plot_losses
# ជាជម្រើស៖
# from llms_from_scratch.ch05 import plot_losses


epochs_tensor = torch.linspace(1, n_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)
plt.tight_layout(); plt.savefig("3.pdf")
plt.show()

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ